# Подсчет уникальных ИНН SA на 01.02.2026

Отдельная тетрадка для ручной проверки из исходных таблиц Озера:
1. Точный подсчет уникальных ИНН клиентов с действующим SA-договором на 01.02.2026.
2. Быстрый sanity-check через `NDV`.
3. Сравнение множества ИНН Озера и Excel (common / only_excel / only_lake).

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
as_of_date = '2026-02-01'
excel_path = '/home/jovyan/documents/Equaring/Проверки/отчет_февраль_2026.xlsx'
excel_inn_col = 'ИНН'

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
def normalize_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    return s if re.fullmatch(r"\d+", s) else None

## 1) Точный подсчет уникальных ИНН SA на 01.02.2026

In [ ]:
with imp:
    sa_inn_exact = imp.fetch(f"""
        select count(*) as uniq_inn_sa_active_on_2026_02_01
        from (
            select cast(c.c_inn as string) as inn
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            where a.acq_class = 'SA'
              and cast(a.d_valid_from as date) <= cast('{as_of_date}' as date)
              and (
                    a.d_valid_to is null
                    or cast(a.d_valid_to as date) >= cast('{as_of_date}' as date)
                  )
              and c.c_inn is not null
            group by cast(c.c_inn as string)
        ) s
    """)

display(sa_inn_exact)

## 2) Быстрый sanity-check через NDV

In [ ]:
with imp:
    sa_inn_ndv = imp.fetch(f"""
        select ndv(cast(c.c_inn as string)) as approx_uniq_inn_sa_active_on_2026_02_01
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where a.acq_class = 'SA'
          and cast(a.d_valid_from as date) <= cast('{as_of_date}' as date)
          and (
                a.d_valid_to is null
                or cast(a.d_valid_to as date) >= cast('{as_of_date}' as date)
              )
          and c.c_inn is not null
    """)

exact_n = int(sa_inn_exact.iloc[0, 0]) if len(sa_inn_exact) else 0
ndv_n = int(sa_inn_ndv.iloc[0, 0]) if len(sa_inn_ndv) else 0

display(pd.DataFrame([
    {'metric': 'exact', 'value': exact_n},
    {'metric': 'ndv_approx', 'value': ndv_n},
    {'metric': 'abs_diff', 'value': abs(exact_n - ndv_n)},
]))

## 3) Сравнение множества ИНН Озера и Excel

In [ ]:
df_excel = pd.read_excel(excel_path)
if excel_inn_col not in df_excel.columns:
    raise ValueError(f'В Excel не найдена колонка: {excel_inn_col}')

excel_inn_set = set(df_excel[excel_inn_col].apply(normalize_id).dropna().tolist())

with imp:
    sa_inn_list = imp.fetch(f"""
        select cast(c.c_inn as string) as inn
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where a.acq_class = 'SA'
          and cast(a.d_valid_from as date) <= cast('{as_of_date}' as date)
          and (
                a.d_valid_to is null
                or cast(a.d_valid_to as date) >= cast('{as_of_date}' as date)
              )
          and c.c_inn is not null
        group by cast(c.c_inn as string)
    """)

lake_inn_set = set(sa_inn_list['inn'].dropna().astype(str).str.strip().tolist())

compare_summary = pd.DataFrame([
    {'metric': 'lake_sa_inn_cnt', 'value': len(lake_inn_set)},
    {'metric': 'excel_inn_cnt', 'value': len(excel_inn_set)},
    {'metric': 'common_inn_cnt', 'value': len(lake_inn_set & excel_inn_set)},
    {'metric': 'only_excel_inn_cnt', 'value': len(excel_inn_set - lake_inn_set)},
    {'metric': 'only_lake_inn_cnt', 'value': len(lake_inn_set - excel_inn_set)},
])

display(compare_summary)
display(pd.DataFrame({'only_excel_inn_sample': sorted(list(excel_inn_set - lake_inn_set))[:100]}))
display(pd.DataFrame({'only_lake_inn_sample': sorted(list(lake_inn_set - excel_inn_set))[:100]}))